# Slide generation across deck types and styles

A board needs to compare investments. New staff need a lesson they can follow. A service team needs to explain an outage without turning an untested diagnosis into a fact. Each task may end in a slide deck, but the useful presentation looks different.

Consider a stockroom lesson. The reorder point is daily demand multiplied by delivery time, plus safety stock. A good slide makes that rule easy to follow, shows the units and gives a practice question before revealing its answer. If demand changes, the worked example, order decision and practice answer must still agree. Making the page attractive is only part of the job.

A model must choose what to emphasize, arrange editable objects, preserve the data, inspect what PowerPoint actually renders, and revise the whole deck. A **harness** is the surrounding software that gives it those tools and passes information between steps. We test whether optional layout helpers and a chance to see its rendered draft improve the final result.

The earlier two-brief pilot favored GPT over Claude and Muse. Screenshot review won 61.1% of paired credit; helpers won 38.2%. That evidence came from closely related financial decks. This follow-up asks whether those rankings survive six deck families and six visual styles.

We use 12 fresh briefs, three producers and four harness conditions, for 144 final eight-slide decks. Three frontier judges compare matching decks using screenshots alone, in both presentation orders. The inputs and 288 pairings were frozen before generation. Results below are replayed from saved responses; this notebook makes no model calls.

**Collection status:** collection stopped at the disk-space guard after 72 draft calls and 111 of 144 revision calls, including one timed-out revision. Paired judging has not started. The analysis below requires the completed evidence and has not yet been executed. Styles are balanced across briefs, not crossed within the same brief. Differences by style are descriptive, not proof that a style caused a change.

## Contents

1. [Setup and saved evidence](#1.-Setup-and-saved-evidence)
2. [The briefs and styles](#2.-The-briefs-and-styles)
3. [How the comparison works](#3.-How-the-comparison-works)
4. [Which models win](#4.-Which-models-win)
5. [What the harness changes](#5.-What-the-harness-changes)
6. [Where the rankings change](#6.-Where-the-rankings-change)
7. [How much the judges agree](#7.-How-much-the-judges-agree)
8. [Slide examples and errors](#8.-Slide-examples-and-errors)
9. [Spend and limits](#9.-Spend-and-limits)
10. [How to reproduce](#10.-How-to-reproduce)
11. [Future work and further reading](#11.-Future-work-and-further-reading)

```mermaid
flowchart TD
    A[Python facts + independent Qwen wording<br/>12 briefs, 6 families, 6 styles] --> B[Freeze facts, revision requests and 288 pairs]
    B --> C[3 producers x 2 interfaces x 12 briefs<br/>72 draft JSON scenes]
    C --> D[Native PowerPoint compiler<br/>72 editable drafts and screenshots]
    D --> E[Two revisions from each draft<br/>text only or text + screenshots]
    E --> F[144 final eight-slide decks<br/>1152 final slide images if all render]
    F --> G[288 matched pairs x 3 judges x 2 orders<br/>1728 planned screenshot-only judgments]
    G --> H[Average within each brief<br/>model ranks, helper effects, review effects]
```

For the stockroom lesson, the initial calculation goes to each producer twice: once with basic drawing tools and once with optional helpers. Each draft then branches into two revision attempts with the same new numbers. Only one branch sees the draft images. Judges compare the resulting lessons against images of the final facts.

## 1. Setup and saved evidence

The code reads local files and checks their identities before calculating results. A cache stores each full request and raw response, including failures. It prevents an offline replay from quietly asking a newer model the same question.

The analysis treats a brief as the unit of evidence. Six votes on one pair are six views of the same two decks, not six independently generated examples. The experiment has twelve briefs and one draft per producer/interface/brief.

In [ ]:
from pathlib import Path
import json, hashlib, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

# Analysis knobs: these do not spend money or contact any service.
N_BOOTSTRAPS = 10000
SEED = 20261022
DATA, FIG = Path('data'), Path('figures')
FIG.mkdir(exist_ok=True)
def read(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def canonical(x): return json.dumps(x, sort_keys=True, ensure_ascii=False, separators=(',', ':'))
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def save(name, obj): (DATA/name).write_text(json.dumps(obj,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
def say(text): display(Markdown(text))
def table(frame):
    vals=frame.astype(str)
    return '| '+' | '.join(map(str,vals.columns))+' |\n| '+' | '.join(['---']*len(vals.columns))+' |\n'+'\n'.join('| '+' | '.join(row)+' |' for row in vals.values)
def finish(name):
    plt.tight_layout();plt.savefig(FIG/name,dpi=180,bbox_inches='tight');plt.show()
plt.rcParams.update({'figure.dpi':110,'font.size':11,'axes.spines.top':False,'axes.spines.right':False,
                     'axes.titlesize':14,'axes.labelsize':11,'figure.facecolor':'white','axes.grid':False})
COLORS={'gpt':'#007F86','claude':'#AE572B','muse':'#6951A5'}
LABELS={'gpt':'GPT 6.1 Sol','claude':'Claude Opus 5.5','muse':'Muse Wonder'}
CONDS=['primitive_text','primitive_vision','helpers_text','helpers_vision']
CLABELS=['Primitives + text','Primitives + screenshots','Helpers + text','Helpers + screenshots']
protocol=read(DATA/'protocol.json');cases=read(DATA/'cases.json');styles=read(DATA/'styles.json')
case_map={c['id']:c for c in cases}
assert sha(DATA/'cases.json')==protocol['cases_sha256']
assert sha(DATA/'styles.json')==protocol['styles_sha256']
assert sha(DATA/'harness_at_freeze.py')==protocol['source_sha256']
evidence_manifest=read(DATA/'manifest.json')
assert all(sha(path)==expected for path,expected in evidence_manifest.items()), 'Saved evidence changed'
runs=read(DATA/'runs.json');run_map={r['artifact']:r for r in runs}
judgments=read(DATA/'paired_judgments.json');pairs=protocol['pairs'];pair_map={p['id']:p for p in pairs}
raw=[json.loads(s) for s in (DATA/'cache.jsonl').read_text(encoding='utf-8').splitlines()]
cache={r['key']:r for r in raw}
assert len(cache)==len(raw) and len(runs)==144 and len(pairs)==288
assert len(judgments)==1728 and len({(r['pair_id'],r['judge'],r['reverse']) for r in judgments})==1728
assert all(hashlib.sha256(canonical(r['request']).encode()).hexdigest()==r['key'] for r in raw)
assert all(j['status'] in ('ok','delivery') for j in judgments), 'Incomplete judge panel: report failures before claiming a complete study'
image_hashes={}
for row in raw:
    for im in row['request']['images']:
        if im['path'] not in image_hashes:image_hashes[im['path']]=sha(im['path'])
        assert image_hashes[im['path']]==im['sha256']
assert read(DATA/'controls.json')['all_passed']
for j in judgments:
    if j['status']!='ok':continue
    row=cache[j['request_key']]
    assert row['request']['prompt']==protocol['prompts']['pair']
    answer=row['response']['text'].strip()
    if answer.startswith('```') and answer.endswith('```'):answer=re.sub(r'^```(?:json)?\s*','',answer)[:-3].strip()
    assert json.loads(answer)==j['parsed'], 'Parsed judgment differs from the raw response'
    p=pair_map[j['pair_id']];ordered=[p['a'],p['b']]
    if j['reverse']:ordered.reverse()
    expected=[f"data/tasks/{p['case']}-{i}.png" for i in range(1,4)]+[im['path'] for artifact in ordered for im in run_map[artifact]['images']]
    assert [im['path'] for im in row['request']['images']]==expected, 'Pair image order differs from the plan'
    mapped={'A':ordered[0],'B':ordered[1],'tie':'tie'}
    assert j['winner']==mapped[j['parsed']['winner']], 'Winner order mapping is inconsistent'
previous=read(DATA/'previous_pilot.json')
say(f"Loaded **{len(runs)} final decks**, {len(pairs)} matched pairs and {len(judgments)} planned comparison records. All six image controls passed. The cache contains {len(raw)} evaluation requests.")

## 2. The briefs and styles

The six families ask for different kinds of communication. A research summary needs caveats and comparisons; a lesson needs a sequence and a delayed answer; an incident review needs to separate observations from a hypothesis. Each family has two numerical variants and two audience descriptions. Both variants keep that family's section structure, so they are related cases.

Python creates the numbers and calculates the expected revisions. Qwen, an open-weight model from a vendor outside the judging panel, supplies the audience framing and visual styles. It receives no numbers when writing the final framing. Earlier authoring drafts that omitted sections or mixed initial and final facts remain in the cache, but are not test cases. This reduces one source of vendor overlap; the fixture code and experiment design were still written with GPT assistance.

Styles rotate across families. Each appears twice, and each family appears in two styles. Holding the source facts fixed across all four harness conditions makes the helper and screenshot comparisons fair within each brief. The design does **not** compare two styles on the same brief, so a weak result under a style could reflect that brief's content instead.

In [ ]:
brief_table=pd.DataFrame([{'ID':c['id'],'Deck family':c['family'].replace('_',' '),'Style':c['style']['id'].replace('_',' '),'Audience':c['audience']} for c in cases])
display(brief_table)
style_table=pd.DataFrame([{'Style':s['id'].replace('_',' '),'Instruction':s['instruction']} for s in styles])
display(style_table)
example=case_map['02-training_lesson-0']
say('The measured training case `02-training_lesson-0` changes these facts:')
say(table(pd.DataFrame([{'Block':k,'Draft facts':v,'Final facts':example['final_blocks'][k]} for k,v in example['initial_blocks'].items() if v!=example['final_blocks'][k]])))
say('With daily demand 19 units, lead time 5 days and safety stock 28 units, the revised reorder point is **123 units**. Inventory position is 118, so the worked example says ORDER; the separate practice position is 135, so its answer is WAIT. Those are different examples, not a contradiction.')
provenance=read(DATA/'date_provenance.json')
display(pd.DataFrame(provenance['models']))
say('Every brief was created on 2026-10-22, after the latest recorded model boundary, Muse’s 2026-10-12 release. Model date sources were checked in the preceding pilot on 2026-10-21 and are retained. Requested CLI aliases can change, so these results describe the recorded CLI systems.')

## 3. How the comparison works

The primitive baseline lets a model place text, rectangles, straight connectors, native charts and native tables. It specifies coordinates, fonts and colors in a JSON scene, a text description that the compiler turns into editable PowerPoint objects. The model chooses the narrative and positions; the compiler does not repair factual mistakes or optimize layout.

The helper condition adds four optional functions: `header`, `kpi`, `flow` and `columns`. They save repetitive geometry work and expand into the same primitives. The repaired header measures text and reserves space for a subtitle. Helpers can also constrain a design: a fixed left-aligned header may fit an executive style better than a centered editorial heading. Models may ignore helpers and draw directly.

After rendering, both revision branches get the same requested factual changes and their draft scene. The screenshot branch also gets eight actual draft images. This tests the benefit of seeing the result within one fixed revision opportunity. It does not test unlimited iterative editing.

```mermaid
flowchart TD
    T[One brief + one interface] --> G[GPT 6.1 Sol via codex exec<br/>draft JSON]
    T --> C[Claude Opus 5.5 via claude -p<br/>draft JSON]
    T --> M[Muse Wonder via muse exec<br/>draft JSON]
    G --> R[Wait for drafts; compile and render serially]
    C --> R
    M --> R
    R --> A[Same producer CLI<br/>draft JSON + final facts → revised JSON]
    R --> B[Same producer CLI<br/>draft JSON + final facts + 8 images → revised JSON]
    A --> F[Compile final decks; assemble matching pairs]
    B --> F
    F --> J1[GPT 6 Astra via codex exec<br/>A/B/tie + visible reasons]
    F --> J2[Claude Opus 5.5 via claude -p<br/>A/B/tie + visible reasons]
    F --> J3[Muse Wonder via muse exec<br/>A/B/tie + visible reasons]
    J1 --> O[Repeat each pair in reverse order in a fresh session]
    J2 --> O
    J3 --> O
```

Each judge receives three task-card images and sixteen slide images. It sees no model names, source JSON or other judges' scores. Five dimensions have equal stated importance: design, legibility, visual communication, coherence and requirements. Requested style informs design; a separate style comparison is diagnostic. The overall choice is the judge's considered tradeoff, not a numerical sum imposed afterward.

**Win credit** assigns 1 to a win, 0.5 to a tie and 0 to a loss. If one judge chooses a deck in one order and ties it in the other, its credit is 0.75. We average orders, then judges, then briefs. A 60% result means 60% of available comparison credit against these alternatives; it is not a 6/10 quality score or a promise that 60% of users will prefer it.

There are 24 pairings per brief: twelve model comparisons across the four conditions, six helper comparisons and six screenshot comparisons. A failed deck loses to a delivered deck before judging. Two failures tie. We report these delivery outcomes separately and repeat the quality comparison using only pairs where both decks rendered.

In [ ]:
first_pair=next(j for j in judgments if j['status']=='ok')
request=cache[first_pair['request_key']]
say('Exact screenshot-only judging prompt:')
print(request['request']['prompt'])
say('One raw judge response, followed by the parsed result:')
print(request['response']['text'])
display(pd.DataFrame([{'Pair':first_pair['pair_id'],'Judge':first_pair['judge'],'Reversed order':first_pair['reverse'],'Winner artifact':first_pair['winner']}]))
walk_pair=pair_map[first_pair['pair_id']]
walk_deck=run_map[walk_pair['a']]
draft_tag=f"{walk_deck['case']}/{walk_deck['model']}/{'helpers' if walk_deck['helpers'] else 'primitive'}/draft"
first_draft=next(r for r in raw if r['request']['tag']==draft_tag)
say('Exact producer prompt for one draft (the interface instructions and brief are both included):')
print(first_draft['request']['prompt'])
say('Its raw scene response is retained in the cache. The opening excerpt shows the scene format:')
print(first_draft['response']['text'][:1500])
walk_revision=cache[walk_deck['request_key']]
say('The corresponding revision receives the same source task, the final facts and this draft. Its full request is retained under `'+walk_deck['request_key']+'`. This branch receives '+str(len(walk_revision['request']['images']))+' draft screenshots.')
walk_votes=[j for j in judgments if j['pair_id']==first_pair['pair_id']]
walk_credit=[.5 if j['winner']=='tie' else float(j['winner']==walk_pair['a']) for j in walk_votes]
say('For this matched pair, credits for the first listed artifact are '+', '.join(f'{x:g}' for x in walk_credit)+f'. Their mean is **{np.mean(walk_credit):.1%}**. That one pair then contributes to its brief average, with the same weight as other briefs.')
rows=[]
for j in judgments:
    p=pair_map[j['pair_id']]
    ca=0.5 if j['winner']=='tie' else float(j['winner']==p['a'])
    rows.append({**p,'judge':j['judge'],'reverse':j['reverse'],'status':j['status'],'credit_a':ca,
                 'family':case_map[p['case']]['family'],'style':case_map[p['case']]['style']['id']})
votes=pd.DataFrame(rows)
assert votes.groupby(['id','judge']).size().eq(2).all()
pairmeans=votes.groupby(['id','case','kind','family','style'],as_index=False)['credit_a'].mean()
model_rows=[];factor_rows=[]
for r in rows:
    if r['kind']=='model':
        external=r['judge'] not in (r['model_a'],r['model_b'])
        for model,credit in [(r['model_a'],r['credit_a']),(r['model_b'],1-r['credit_a'])]:
            model_rows.append({**r,'model':model,'credit':credit,'external':external})
    else:
        factor_rows.append({**r,'effect':r['kind'],'setting':r.get('review',r.get('interface')),'credit':1-r['credit_a'],'external':r['judge']!=r['model']})
mv=pd.DataFrame(model_rows);fv=pd.DataFrame(factor_rows)
for name,frame in [('votes.csv',votes),('model_votes.csv',mv),('factor_votes.csv',fv)]:frame.to_csv(DATA/name,index=False)
case_ids=[c['id'] for c in cases]
rng=np.random.default_rng(SEED)
family_indices=[[i for i,c in enumerate(cases) if c['family']==f] for f in dict.fromkeys(c['family'] for c in cases)]
bootstrap=np.concatenate([rng.choice(ix,(N_BOOTSTRAPS,len(ix)),replace=True) for ix in family_indices],axis=1)
def estimate(frame,groups):
    out=[]
    for key,part in frame.groupby(groups):
        if not isinstance(key,tuple):key=(key,)
        means=part.groupby('case')['credit'].mean().reindex(case_ids)
        values=means.to_numpy()
        if np.isfinite(values).all():lo,hi=np.quantile(values[bootstrap].mean(axis=1),[.025,.975])
        else:lo=hi=np.nan
        out.append(dict(zip(groups,key),credit=means.mean(),low=lo,high=hi,briefs=int(means.notna().sum())))
    return pd.DataFrame(out)
models=estimate(mv,['condition','model'])
factors=estimate(fv,['effect','setting'])
models.to_csv(DATA/'models.csv',index=False);factors.to_csv(DATA/'factors.csv',index=False)

## 4. Which models win

Compare models within each harness column. Each producer faces the other two producers on the same twelve briefs. The dashed line marks 50% win credit, the point at which wins and losses balance after ties receive half credit.

The intervals resample whole briefs, keeping their votes and all conditions together. We resample two briefs within each family, 10,000 times, and show the middle 95% of the resulting means. These intervals describe variation within this small, deliberately chosen suite. They cannot capture all possible presentation topics or repeated-generation variability.

In [ ]:
leaders={cond:models[models.condition==cond].sort_values('credit',ascending=False).iloc[0]['model'] for cond in CONDS}
leading_counts=pd.Series(list(leaders.values())).value_counts()
fig,axes=plt.subplots(1,4,figsize=(15,4),sharey=True)
for ax,cond,label in zip(axes,CONDS,CLABELS):
    subset=models[models.condition==cond].set_index('model').loc[list(LABELS)]
    for i,(m,r) in enumerate(subset.iterrows()):
        ax.errorbar(i,100*r.credit,yerr=[[100*(r.credit-r.low)],[100*(r.high-r.credit)]],fmt='o',color=COLORS[m],capsize=4,markersize=8)
    ax.axhline(50,color='#999999',linestyle='--');ax.set_xticks(range(3),['GPT','Claude','Muse']);ax.set_ylim(-3,103);ax.set_title(label,fontsize=12)
axes[0].set_ylabel('Win credit (%)')
fig.suptitle(f'{LABELS[leading_counts.index[0]]} leads in {leading_counts.iloc[0]} of four harness conditions',y=1.04)
finish('model_rankings.png')
mt=models.pivot(index='model',columns='condition',values='credit').loc[list(LABELS),CONDS]
say(table((mt*100).round(1).rename(index=LABELS,columns=dict(zip(CONDS,CLABELS))).reset_index()))
leaders={cond:models[models.condition==cond].sort_values('credit',ascending=False).iloc[0]['model'] for cond in CONDS}
say('Condition leaders: '+ '; '.join(f"{CLABELS[CONDS.index(c)]}: {LABELS[m]}" for c,m in leaders.items())+'. Each mean covers twelve briefs and two opponents per brief. A leader’s interval should be read alongside the matched gaps and family breakdowns, rather than as a universal ranking.')
old=pd.DataFrame(previous['models.csv']);old['win_credit']=old.win_credit.astype(float)
compare=models.merge(old[['condition','producer','win_credit']],left_on=['condition','model'],right_on=['condition','producer'])
compare['change_pp']=100*(compare.credit-compare.win_credit)
display(compare[['condition','model','credit','win_credit','change_pp']].round(3))
matched_gaps=[]
for cond in CONDS:
    part=mv[mv.condition==cond].groupby(['case','model']).credit.mean().unstack().reindex(case_ids)
    gap=part.gpt-part.claude
    lo,hi=np.quantile(gap.to_numpy()[bootstrap].mean(axis=1),[.025,.975])
    matched_gaps.append({'condition':cond,'GPT_minus_Claude_pp':100*gap.mean(),'low':100*lo,'high':100*hi})
display(pd.DataFrame(matched_gaps).round(1))
direct=estimate(votes[votes.kind=='model'].rename(columns={'credit_a':'credit'}),['condition','model_a','model_b'])
direct.to_csv(DATA/'direct_models.csv',index=False)
say('Direct matchups keep the opponent fixed. Credit in this table belongs to model A against model B; it is not the average against both other producers.')
display(direct.round(3))
closest=direct.iloc[(direct.credit-.5).abs().argmin()]
say(f"The closest direct matchup is {LABELS[closest.model_a]} versus {LABELS[closest.model_b]} under {closest.condition}: {closest.credit:.1%} credit for the first model across twelve briefs. Such close comparisons deserve more weight in a practical choice than a large margin against a weaker opponent.")
rank_orders={cond:models[models.condition==cond].sort_values('credit',ascending=False)['model'].tolist() for cond in CONDS}
previous_orders={cond:old[old.condition==cond].sort_values('win_credit',ascending=False)['producer'].tolist() for cond in CONDS}
order_matches={cond:rank_orders[cond]==previous_orders[cond] for cond in CONDS}
say(f"The complete three-model ordering matches the earlier pilot in **{sum(order_matches.values())}/4** conditions. The earlier order was GPT, Claude, Muse in every condition; the new orders are "+'; '.join(CLABELS[CONDS.index(c)]+': '+', '.join(LABELS[m] for m in rank_orders[c]) for c in CONDS)+'. These are point-estimate ranks; the matched gaps show their uncertainty.')
save('ranking_summary.json',{'leaders':leaders,'orders':rank_orders,'orders_match_previous':order_matches,'matched_gaps':matched_gaps,'previous_comparison':compare.to_dict('records')})
say('The preceding pilot used two financial briefs. Its numbers are shown separately and are not added to this sample. A change in credit can reflect new opponents’ outputs, deck content, styles or model service drift; it is not a causal estimate of any one change.')

## 5. What the harness changes

This is the **ablation**, a comparison that adds or removes one component while keeping the rest fixed. To test helpers, we compare the same producer and brief with and without helpers, separately for text review and screenshot review. To test screenshots, we compare the two revisions of the same draft, separately for primitives and helpers.

The horizontal axis below is credit for the added feature. Values above 50% favor adding it. Values below 50% favor the simpler setup. The two helper contrasts come from independently generated drafts; the screenshot contrasts share a draft, which controls that source of variation more tightly.

In [ ]:
factors['label']=factors.apply(lambda r:('Add helpers, '+('screenshots' if r.setting=='vision' else 'text review')) if r.effect=='helpers' else 'Add screenshots, '+r.setting,axis=1)
fig,ax=plt.subplots(figsize=(9,4))
for i,r in factors.reset_index(drop=True).iterrows():
    ax.errorbar(100*r.credit,i,xerr=[[100*(r.credit-r.low)],[100*(r.high-r.credit)]],fmt='o',color='#007F86',capsize=4,markersize=8)
ax.axvline(50,color='#999999',linestyle='--');ax.set_xlim(0,100);ax.set_yticks(range(len(factors)),factors.label);ax.set_xlabel('Win credit for the added feature (%)');ax.set_title(f'Added features exceed 50% credit in {(factors.credit>0.5).sum()}/4 contrasts')
finish('harness_effects.png')
say(table(factors[['label','credit','low','high']].assign(credit=lambda x:(100*x.credit).round(1),low=lambda x:(100*x.low).round(1),high=lambda x:(100*x.high).round(1))))
pooled=estimate(fv,['effect']);save('harness_summary.json',{'pooled':pooled.to_dict('records'),'by_setting':factors.to_dict('records')})
for _,r in pooled.iterrows():
    oldvalue=previous['summary.json']['helper_win_credit' if r.effect=='helpers' else 'screenshot_win_credit']
    say(f"{r.effect.capitalize()} receive **{r.credit:.1%}** credit pooled across the other factor, compared with {oldvalue:.1%} in the preceding pilot. The new estimate averages twelve briefs and all three producers.")
fm=fv.groupby(['effect','setting','model']).credit.mean().unstack()
display((100*fm).round(1).rename(columns=LABELS))
interaction=[]
for effect,left,right in [('helpers','text','vision'),('vision','primitive','helpers')]:
    q=fv[fv.effect==effect].groupby(['case','setting']).credit.mean().unstack().reindex(case_ids)
    delta=q[right]-q[left];lo,hi=np.quantile(delta.to_numpy()[bootstrap].mean(axis=1),[.025,.975])
    interaction.append({'effect':effect,'comparison':right+' minus '+left,'difference_pp':100*delta.mean(),'low':100*lo,'high':100*hi})
display(pd.DataFrame(interaction).round(1));save('interactions.json',interaction)
say('The difference between feature credits describes how preferences depend on the other feature. It is not an additive change on an underlying quality scale. Model-specific rows matter when choosing a harness: an average benefit can hide a producer that uses a tool poorly.')
helper_usage=[]
for r in runs:
    path=DATA/'artifacts'/r['artifact']/'scene.json'
    if not path.exists():continue
    scene=read(path)
    types=[e.get('type') for slide in scene.get('slides',[]) for e in slide.get('elements',[])]
    helper_usage.append({'artifact':r['artifact'],'case':r['case'],'model':r['model'],'condition':r['condition'],'offered':r['helpers'],
                         **{k:types.count(k) for k in ['header','kpi','flow','columns']}})
helper_usage=pd.DataFrame(helper_usage)
helper_usage['any_used']=helper_usage[['header','kpi','flow','columns']].sum(axis=1)>0
usage_summary=helper_usage[helper_usage.offered].groupby('model').agg(decks=('artifact','size'),used_any=('any_used','sum'),headers=('header','sum'),kpis=('kpi','sum'),flows=('flow','sum'),columns=('columns','sum'))
display(usage_summary)
helper_usage.to_csv(DATA/'helper_usage.csv',index=False)
say('Helpers are optional. This table counts actual calls in the final scenes. The controlled comparison is about offering the helper interface; a deck in that condition can still be built entirely from primitives. Usage counts help explain adoption but do not identify the causal effect of choosing a particular helper.')

## 6. Where the rankings change

An average may hide a deck family that reverses the result. The first heatmap shows model credit by family, averaged across harness conditions. The second shows credit for adding each harness component. Each row contains only two briefs, so use it to identify follow-ups, not to declare a category winner.

We also leave out one family at a time and recompute each condition's model leader. If removing a family changes the leader, the headline depends on the mix of tasks. The style table uses the same data in another grouping; content and style remain mixed together.

In [ ]:
family_models=mv.groupby(['family','model']).credit.mean().unstack().loc[:,list(LABELS)]
family_factors=fv.groupby(['family','effect']).credit.mean().unstack()
fig,axes=plt.subplots(1,2,figsize=(12,6),gridspec_kw={'width_ratios':[3,2]})
for ax,frame,title in [(axes[0],family_models,'Model credit across harness conditions'),(axes[1],family_factors,'Credit for the added feature')]:
    ax.imshow(frame*100,vmin=0,vmax=100,cmap='YlGnBu',aspect='auto')
    ax.set_xticks(range(len(frame.columns)),[LABELS.get(c,c) for c in frame.columns],rotation=25,ha='right')
    ax.set_yticks(range(len(frame)),[s.replace('_',' ') for s in frame.index]);ax.set_title(title,fontsize=12)
    for y in range(len(frame)):
        for x in range(len(frame.columns)):
            v=100*frame.iloc[y,x];ax.text(x,y,f'{v:.1f}%',ha='center',va='center',color='white' if v>65 else 'black')
family_leaders=family_models.idxmax(axis=1).value_counts()
fig.suptitle(f'{LABELS[family_leaders.index[0]]} leads in {family_leaders.iloc[0]}/6 families, averaging harness conditions',y=1.03)
finish('family_results.png')
say(table((family_models*100).round(1).rename(columns=LABELS).reset_index()))
say(table((family_factors*100).round(1).reset_index()))
loo=[]
for family in family_models.index:
    q=mv[mv.family!=family].groupby(['condition','model']).credit.mean()
    for cond in CONDS:
        top=q.loc[cond].idxmax();loo.append({'Excluded family':family,'Condition':cond,'Leader':LABELS[top],'Credit':round(100*q.loc[cond,top],1),'Same leader':top==leaders[cond]})
loo=pd.DataFrame(loo);display(loo)
say(f"The condition leader stays unchanged in {int(loo['Same leader'].sum())}/{len(loo)} leave-one-family-out comparisons. These are sensitivity checks on the same data, not new replications.")
style_models=mv.groupby(['style','model']).credit.mean().unstack()
style_factors=fv.groupby(['style','effect']).credit.mean().unstack()
display((100*style_models).round(1));display((100*style_factors).round(1))
save('variation_summary.json',{'same_leader':int(loo['Same leader'].sum()),'checks':len(loo),'family_models':family_models.reset_index().to_dict('records'),'family_factors':family_factors.reset_index().to_dict('records'),'style_models':style_models.reset_index().to_dict('records'),'style_factors':style_factors.reset_index().to_dict('records')})

## 7. How much the judges agree

Swapping the two decks checks sensitivity to presentation order. A changed verdict includes a win becoming a tie, not only a complete reversal. We show each judge separately because averaging can hide different standards.

For a model comparison, the other-vendor sensitivity uses only the judge from neither producer's vendor. For a helper or screenshot comparison, it excludes the producer's vendor. This reduces direct vendor overlap but leaves a smaller panel. It does not turn model preferences into human judgments.

The judges also list unmet requirements. A deck appears in several pairs, so we first average its reported failure rate within each judge and then across judges. Repeated appearances do not create additional decks. The checks bundle several facts per source block; they are a useful diagnostic, not a verified factual-accuracy score.

In [ ]:
valid=votes[votes.status=='ok']
order=valid.pivot(index=['id','judge'],columns='reverse',values='credit_a').dropna()
order['changed']=order[False]!=order[True]
tie_rate=float((valid.credit_a==.5).mean())
judge_models=mv.groupby(['judge','model']).credit.mean().unstack().loc[:,list(LABELS)]
fig,axes=plt.subplots(1,2,figsize=(12,4))
judge_models.rename(columns=LABELS).mul(100).plot.bar(ax=axes[0],color=[COLORS[m] for m in LABELS],rot=0)
axes[0].set_ylabel('Win credit (%)');axes[0].set_ylim(0,100);axes[0].set_title('Each judge’s producer preferences');axes[0].legend(fontsize=8)
order.groupby('judge').changed.mean().mul(100).plot.bar(ax=axes[1],color='#AE572B',rot=0)
axes[1].set_ylabel('Changed verdict after order swap (%)');axes[1].set_ylim(0,100);axes[1].set_title(f'Verdicts change in {order.changed.mean():.1%} of swaps')
finish('judge_reliability.png')
say(f"Judges changed their verdict in **{int(order.changed.sum())}/{len(order)}** swapped-order checks ({order.changed.mean():.1%}). Ties account for **{tie_rate:.1%}** of valid judgments. These rates measure repeat consistency, not correctness.")
external_models=estimate(mv[mv.external],['condition','model']);external_factors=estimate(fv[fv.external],['effect','setting'])
display(external_models.round(3));display(external_factors.round(3))
quality_models=estimate(mv[mv.status=='ok'],['condition','model']);quality_factors=estimate(fv[fv.status=='ok'],['effect','setting'])
display(quality_models.round(3));display(quality_factors.round(3))
check_rows=[];dimension_rows=[]
for j in judgments:
    if j['status']!='ok':continue
    p=pair_map[j['pair_id']]
    for artifact,unmet in j['unmet'].items():
        r=run_map[artifact]
        check_rows.append({'artifact':artifact,'case':r['case'],'model':r['model'],'condition':r['condition'],'judge':j['judge'],'rate':len(unmet)/14,'unmet':unmet})
    if p['kind']!='model':
        for dim,w in j['dimensions'].items():
            dimension_rows.append({'effect':p['kind'],'setting':p.get('review',p.get('interface')),'dimension':dim,'credit':.5 if w=='tie' else float(w==p['b'])})
checks=pd.DataFrame(check_rows)
deck_checks=checks.groupby(['artifact','case','model','condition','judge']).rate.mean().groupby(['artifact','case','model','condition']).mean().reset_index()
check_table=deck_checks.groupby(['model','condition']).rate.mean().unstack().loc[:,CONDS]
dimension_table=pd.DataFrame(dimension_rows).groupby(['effect','setting','dimension']).credit.mean().unstack()
display((100*check_table).round(1));display((100*dimension_table).round(1))
delivery=pd.DataFrame(runs).groupby(['model','condition']).status.agg(total='size',rendered=lambda s:int((s=='ok').sum())).reset_index()
display(delivery)
drafts=read(DATA/'drafts.json')
draft_ok={(d['case'],d['model'],d['helpers']):d['status']=='ok' for d in drafts}
recovery=pd.DataFrame([{'model':r['model'],'condition':r['condition'],'draft_ok':draft_ok[r['case'],r['model'],r['helpers']],'final_ok':r['status']=='ok'} for r in runs])
failed_start=recovery[~recovery.draft_ok]
if len(failed_start):
    display(failed_start.groupby(['model','condition']).agg(attempts=('final_ok','size'),recovered=('final_ok','sum')))
    say(f"{len(drafts)-sum(draft_ok.values())}/{len(drafts)} drafts failed to render. Their {len(failed_start)} scheduled revisions produced {int(failed_start.final_ok.sum())} valid final decks. These branches received the same compiler error; their screenshot condition had no images to supply. The primary result measures offering screenshot review within this pipeline, including such cases.")
    actual_images=fv[(fv.effect=='vision')&fv.apply(lambda r:draft_ok[r['case'],r['model'],r['interface']=='helpers'],axis=1)]
    display(estimate(actual_images,['effect','setting']).round(3))
    say('This exploratory sensitivity retains screenshot contrasts only when the draft rendered. It checks actual image exposure, with a smaller and selected set of drafts. It does not replace the full scheduled comparison.')
recovery.to_csv(DATA/'delivery_recovery.csv',index=False)
save('reliability_summary.json',{'order_changed':int(order.changed.sum()),'order_checks':len(order),'tie_rate':tie_rate,'valid_judgments':len(valid),'delivery_comparisons':int((votes.status=='delivery').sum()),'valid_decks':sum(r['status']=='ok' for r in runs),'external_models':external_models.to_dict('records'),'external_factors':external_factors.to_dict('records'),'requirement_rates':check_table.reset_index().to_dict('records')})
deck_checks.to_csv(DATA/'deck_checks.csv',index=False)

## 8. Slide examples and errors

The following panels use the first brief in each family, a fixed content-bearing slide and producers in a fixed GPT/Claude/Muse rotation. This selection rule was set before reading the quality judgments. Each panel shows the same producer, facts and slide number under all four conditions. Read across a row for screenshot review; read down a column for helpers.

These examples show the actual rendered outputs, including awkward designs. They help explain what the comparisons mean; they do not replace the results over all twelve briefs. A judge's whole-deck decision can depend on slides outside the panel.

In [ ]:
specs=[('00-board_decision-0','gpt',1),('01-research_brief-0','claude',3),('02-training_lesson-0','muse',7),('03-product_launch-0','gpt',4),('04-incident_review-0','claude',5),('05-community_campaign-0','muse',4)]
examples=[]
for case,model,slide in specs:
    c=case_map[case];block=c['outline'][slide-1]['blocks'][0]
    fig,axes=plt.subplots(2,2,figsize=(18,11))
    chosen={}
    for ax,cond,label in zip(axes.flat,CONDS,CLABELS):
        r=next(r for r in runs if r['case']==case and r['model']==model and r['condition']==cond);chosen[cond]=r['artifact']
        if r['status']=='ok':
            ax.imshow(Image.open(r['images'][slide-1]['path']))
        else:ax.text(.5,.5,'Deck failed to render: '+r.get('error',''),ha='center',va='center',wrap=True)
        ax.axis('off');ax.set_title(label,fontsize=15)
    fig.suptitle(f"{case} | {LABELS[model]} | slide {slide}",fontsize=17)
    name=f'example_{case[:2]}.png';finish(name)
    say('Full-resolution slides: '+' | '.join(f"[{label}]({run_map[chosen[cond]]['images'][slide-1]['path']})" for cond,label in zip(CONDS,CLABELS) if run_map[chosen[cond]]['status']=='ok'))
    evidence=fv[(fv.case==case)&(fv.model==model)].groupby(['effect','setting']).credit.mean()
    summary='; '.join(f"{effect} with {setting}: {credit:.1%}" for (effect,setting),credit in evidence.items())
    say(f"**Expected content:** {c['final_blocks'][block]}\n\n**Measured whole-deck comparisons:** {summary}. These percentages each average six judgments on one matched pair.")
    relevant=[j for j in judgments if j['status']=='ok' and pair_map[j['pair_id']]['case']==case and pair_map[j['pair_id']].get('model')==model]
    # A fixed first pair/judge/order provides an auditable reason, not a selected flattering quote.
    reason=sorted(relevant,key=lambda j:(j['pair_id'],j['judge'],j['reverse']))[0]
    p=pair_map[reason['pair_id']];ordered=[p['a'],p['b']]
    if reason['reverse']:ordered.reverse()
    mapping=', '.join(f"{letter} = {run_map[a]['condition']}" for letter,a in zip('AB',ordered))
    say(f"Judge example ({reason['judge']}, {mapping}): {reason['parsed']['reason']}")
    examples.append({'case':case,'model':model,'slide':slide,'figure':name,'expected':c['final_blocks'][block],'comparison':summary,'judge':reason['judge'],'mapping':mapping,'reason':reason['parsed']['reason'],'pair_id':reason['pair_id']})
save('examples.json',examples)
failed_decks=sorted([r for r in runs if r['status']!='ok'],key=lambda r:(r['case'],r['model'],r['condition']))
if failed_decks:
    failure=failed_decks[0]
    path=DATA/'artifacts'/failure['artifact']/'scene.json'
    if path.exists():
        scene=read(path);counts=[len(s.get('elements',[])) for s in scene.get('slides',[])]
        fig,ax=plt.subplots(figsize=(9,4))
        ax.bar(range(1,len(counts)+1),counts,color=['#AE572B' if n>28 else '#007F86' for n in counts])
        ax.axhline(28,color='black',linestyle='--',label='Frozen limit: 28')
        ax.set_xticks(range(1,len(counts)+1));ax.set_xlabel('Slide number');ax.set_ylabel('Submitted scene objects, before helper expansion')
        ax.set_title(f"A delivery failure submits up to {max(counts)} objects on one slide");ax.legend()
        finish('delivery_failure.png')
        say(f"The first failed final deck by stable case/model/condition order is `{failure['case']}`, {LABELS[failure['model']]}, `{failure['condition']}`. The compiler reports **{failure['error']}**. Its per-slide object counts are {counts}. The exact scene remains saved under `{failure['artifact']}`. It has no final screenshots to judge, so its primary comparisons are delivery outcomes; the quality-only analysis excludes them.")
        save('failure_example.json',{'case':failure['case'],'model':failure['model'],'condition':failure['condition'],'artifact':failure['artifact'],'error':failure['error'],'counts':counts,'total_failed_finals':len(failed_decks)})
    say(f"There are {len(failed_decks)}/144 failed final decks. The 28-object limit applies to submitted scene elements, before helpers expand. A helper can therefore pack several native objects into one submitted element. The helper comparison includes that representational advantage as well as layout assistance.")

## 9. Spend and limits

The collection budget caps the number of calls and output size, and limits each Claude request to a stated dollar amount. All producers receive one draft and one revision opportunity for each final deck. There are no outcome-driven repair attempts. Rendering and file checks happen locally.

Claude's CLI reports a list-price estimate; it is not a subscription invoice. GPT and Muse do not provide comparable dollar accounting here. Missing prices stay missing, rather than being treated as free. Authoring costs include discarded development responses.

In [ ]:
usage=pd.DataFrame([{'phase':r['request']['phase'],'model':r['request']['model']['id'],'vendor':r['request']['model']['vendor'],'seconds':r['response']['elapsed_s'],'reported_usd':r['response'].get('list_cost_usd'),'error':bool(r['response'].get('error'))} for r in raw])
author_raw=[json.loads(s) for s in (DATA/'author_cache.jsonl').read_text(encoding='utf-8').splitlines()]
author_cost=sum((r.get('response',{}).get('usage',{}).get('cost') or 0) for r in author_raw)
claude_cost=float(usage[usage.vendor=='Anthropic'].reported_usd.sum())
display(usage.groupby(['phase','vendor']).agg(calls=('seconds','size'),mean_seconds=('seconds','mean'),errors=('error','sum'),reported_usd=('reported_usd',lambda x:x.sum(min_count=1))).round(3))
fig,ax=plt.subplots(figsize=(8,4))
usage.groupby(['phase','vendor']).seconds.mean().unstack().plot.bar(ax=ax,rot=0)
ax.set_ylabel('Mean elapsed time per request (seconds)');ax.set_title(f'Generation averages {usage[usage.phase=="generation"].seconds.mean():.0f}s per call; judging {usage[usage.phase=="pair"].seconds.mean():.0f}s');finish('call_times.png')
spend={'evaluation_calls':len(raw),'author_calls':len(author_raw),'claude_list_price_usd':claude_cost,'qwen_reported_usd':author_cost,'gpt_usd':None,'muse_usd':None,'errors':int(usage.error.sum())}
save('spend.json',spend);usage.to_csv(DATA/'usage.csv',index=False)
say(f"The run used **{len(raw)} evaluation calls** and {len(author_raw)} fixture-authoring calls. Reported costs are **${claude_cost:.2f} for Claude** and **${author_cost:.4f} for Qwen**. GPT and Muse dollar costs are unavailable. These are partial cost records, not a complete study price.")

Twelve briefs are broader than the original two, but they are still a designed test suite. There are only two variants per family, no repeated draft generations, and no images, logos, photographs or animation. Each deck has eight slides. The brief supplies the slide sequence, so this does not test unrestricted story planning. Conclusions apply to editable, data-led decks produced through this limited scene interface.

The six styles are useful stress tests. They are not a randomized test of theme preference: each style appears on different content. A future within-brief crossover is needed to separate a style's effect from a task's difficulty.

Screenshot judges can inspect visible facts, hierarchy and alignment, but cannot establish native editability or audience usefulness. File checks establish that native objects exist; they do not validate the entire chart's semantics. There are no human raters in this study. Order checks and other-vendor sensitivity reveal some limits of the judge panel, not all of them.

The header helper, primitive compiler, producer models and effort settings match the earlier pilot. The five overall judging dimensions retain equal stated importance; the new prompt also asks for a separate style comparison and unmet-requirement lists. Those added diagnostics could affect judge attention, so differences from the earlier pilot are not a pure estimate of changing the data alone. Concurrency increased from three to six calls. We do not interpret changes in latency across studies as model improvements. Requested model aliases can also change on the provider's side.

## 10. How to reproduce

For an offline replay, install `requirements.txt`, open this notebook in its folder and run every cell. Or run:

```bash
jupyter nbconvert --to notebook --execute --inplace 20_powerpoint_generalization.ipynb
```

The notebook reads saved requests and judgments and rebuilds the analysis figures and tables. It imports no repository helper and makes no network or model calls. The saved PowerPoints and slide images allow inspection without installing PowerPoint.

A fresh collection requires Windows, Microsoft PowerPoint, the three authenticated model CLIs and an OpenRouter key for the independent open-weight fixture author. Use a separate copy and preserve the committed evidence. The top of `harness.py` records the brief count, model IDs, concurrency and call caps. Live calls require `--live`:

```bash
python harness.py prepare --live
python -c "import harness; harness.refine_styles(True)"
python harness.py self-test
python harness.py init
python harness.py run --live
```

The committed run includes documented preflight edits to the fixture format. A new preparation is a new study and must receive its own frozen protocol before any evaluated generation. The CLI collector is separate because PowerPoint rendering and live authentication need an operating-system process. All replay analysis is defined inside this notebook.

## 11. Future work and further reading

Links checked on 2026-10-22.

### Future work

- Cross every brief with two or more styles, holding its facts and slide sequence fixed. This would test whether style itself changes the model or helper ranking; the current style groups each contain different content.
- Repeat draft generation several times per cell and add new families. Measure how often the winning setup changes across runs; the current intervals describe twelve briefs with one draft per producer/interface.
- Add a layout helper that accepts alignment, density and theme parameters, then compare it with these frozen helpers and primitives. The current fixed header and equal columns may fit some requested styles better than others; the style and family results identify where to test this.
- Compare one revision with a fixed two- or three-pass screenshot budget. Measure marginal quality, requirement retention and cost per extra pass; this study measures only the first visual review.
- Raise the 28-object-per-slide limit in a separate controlled run while keeping the same tools and screenshot judges. Measure delivery recovery and paired quality: object-limit failures in this run show that the interface budget can affect the result before visual judging begins.

### Further reading

- [DECKBench](https://arxiv.org/abs/2602.13318) (Jang et al., 2026). Separates slide generation and editing, motivating checks for both layout and content preservation.
- [Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena](https://arxiv.org/abs/2306.05685) (Zheng et al., 2023). Discusses position and self-enhancement biases, which motivate order swaps and vendor sensitivity checks here.
- [Working with charts](https://python-pptx.readthedocs.io/en/latest/user/charts.html) (python-pptx maintainers, accessed 2026). Explains native chart data and why file editability needs a separate check from screenshots.
- [Qwen3.5-397B-A17B](https://huggingface.co/Qwen/Qwen3.5-397B-A17B/tree/main) (Qwen Team, 2026). Documents the open-weight model used to write independent audience framing and style instructions.